# v13 - Round-2 teacher (arm N) and an MRI-pretrained second model (arm M) (GPU T4 x2)

Design: `docs/research/v13-strategy-revision-design.md` revision 3.2, decision D-015.

**Provenance.** Derived from our own `notebooks/v11-oof-teacher.ipynb` (Kaggle
`lingxd/v11-oof-teacher` version 5). The trainer is v11's with explicit changes (see its
docstring). Arm M uses the MRI-CORE weights (Duke, `github.com/mazurowski-lab/mri_foundation`,
Apache 2.0): the released DINOv2 teacher, mapped onto timm's plain ViT-B/16.

- **Target** for every arm: `0.5 * soft + 0.5 * v11 OOF` (the v06 4-source soft target and the v11
  arm-B out-of-fold prediction of the same study, K_infer 16), computed in this notebook; v06 loss
  weights unchanged (0.3 where pilkwang's verdict is UNK); gold studies never trained on.
- **Arms.** N = ConvNeXt-tiny exactly as v11 arm B (320 px, 15 epochs, LR 1e-4 / 3e-4).
  M = MRI-CORE ViT-B/16 at 224 px, CLS token, blocks 8-11 and the final norm trainable, LR 5e-5 /
  3e-4, per-slice min-max input. E = ImageNet EfficientNet-B3 at 320 px with BN in eval mode, LR
  1e-4 / 3e-4 (fallback for M). All: v11-B schedule, K_train 4, K_infer 16, evaluations from epoch
  4, no training-subset loss.

Modes (`MODE` in the configuration cell):

- `smoke` - step S: arms M (GPU 0) and E (GPU 1) in parallel, about 300 micro-steps and one K16
  validation pass each; checks (strict load, gradients, loss decrease, save/reload) and projected
  fold and test-leg times; the admission rule of design 4.2.
- `fold0_arms` - step F0: arm N (GPU 0) and the admitted arm M or E (GPU 1) on fold 0, then Gate F0
  (design 4.4).
- `folds` - steps F1-4: the jobs in `FOLDS_JOBS` ((arm, fold) pairs) through a two-GPU queue.

Budget formula (design 4.1): a job may start only if `projected x 1.15 + 0.25 h` fits before the
session hard stop `QUOTA_AT_LAUNCH_H - 0.4 h`; each arm's time limit is `projected x 1.15`, capped
by the hard stop. `QUOTA_AT_LAUNCH_H` is set from `kaggle quota` right before each push. An arm
that stops early or fails is incomplete.

Inputs: outputs of `lingxd/v06-dicom-audit` and `lingxd/v07-cache-320`, the private datasets
`lingxd/rsna-knee-v11-oof` (v11 OOF and gold predictions) and `lingxd/rsna-knee-mri-core`
(`MRI_CORE_vitb.pth`); internet on for the timm ImageNet weights (training only).
Outputs in `/kaggle/working/v13/`: checkpoints `v13_fold_<k>_<arm>_best.pt`, receipts, OOF and gold
predictions, `v13_train_targets.csv`, decisions.

In [ ]:
# v13 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v13'
MODE = 'fold0_arms'           # 'smoke' (step S), 'fold0_arms' (step F0) or 'folds' (steps F1-4)
ARM_M = 'M'                   # 'fold0_arms': 'M' or 'E' as admitted by step S, or None for N alone
FOLD0_ARMS = ['N', 'M']       # 'fold0_arms': arms to run ('N' and/or ARM_M; a technical rerun may list one)
FOLDS_JOBS = []               # 'folds': list of (arm, fold) pairs, e.g. [('N', 1), ('N', 2)]
QUOTA_AT_LAUNCH_H = 8.57      # GPU quota left right before this push (kaggle quota)
QUOTA_AT_F0_H = 8.25          # 'smoke': quota expected at the F0 launch, for the admission rule
NOTEBOOK_T0 = time.time()
HARD_STOP_S = min(10.5, QUOTA_AT_LAUNCH_H - 0.4) * 3600   # design 4.1
WORK = Path('/kaggle/working') / VERSION
WORK.mkdir(parents=True, exist_ok=True)

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
OOF_WEIGHT = 0.5   # training target = (1 - OOF_WEIGHT) * soft target + OOF_WEIGHT * v11 OOF
OOF_FILES = {k: f'v11_oof_fold{k}_B_k16.csv' for k in range(5)}
V11_GOLD_FOLD0 = 'v11_gold_fold0_B_k16.csv'
MRI_CORE_FILE = 'MRI_CORE_vitb.pth'

COMMON = {
    'version': VERSION,
    'weight_decay': 0.05,
    'warmup_epochs': 1,
    'effective_batch_studies': 8,
    'grad_checkpointing': False,
    'eval_from_epoch': 4,
    'k_infer': 16,
    'train_probe_n': 0,
    'epochs': 15,
    'k_train': 4,
    'micro_batch_studies': 2,
    'smoke_steps': 300,
    'n_test_studies': 1300,
    'eval_batch_tokens': 160,
    'num_workers': 2,
    'seed': 2026,
    'device': 'cuda',
}
# Arms (design 4.2, 4.3). projected_fold_h: N from v11-B fold 0 without its training-subset
# evaluations; M and E are filled in from the step-S receipts before the F0 push.
ARMS = {
    'N': {'kind': 'timm', 'backbone': 'convnext_tiny.fb_in22k_ft_in1k', 'pretrained': True, 'img': 320,
          'norm': 'imagenet', 'lr_backbone': 1e-4, 'lr_head': 3e-4, 'bn_eval': False, 'train_blocks_from': None,
          'projected_fold_h': 6.3},
    'M': {'kind': 'mricore', 'backbone': 'vit_base_patch16_224', 'pretrained': False, 'img': 224,
          'norm': 'minmax', 'lr_backbone': 5e-5, 'lr_head': 3e-4, 'bn_eval': False, 'train_blocks_from': 8,
          'projected_fold_h': 5.07},   # step S (v13 version 1)
    'E': {'kind': 'timm', 'backbone': 'efficientnet_b3.ra2_in1k', 'pretrained': True, 'img': 320,
          'norm': 'imagenet', 'lr_backbone': 1e-4, 'lr_head': 3e-4, 'bn_eval': True, 'train_blocks_from': None,
          'projected_fold_h': 5.92},   # step S (v13 version 1)
}
GATE_S = {'margin': 1.15, 'overhead_h': 0.25, 'reserve_h': 0.4, 'max_reserved_gb': 14.0}
GATE_F0 = {'sanity_gold': 0.900, 'min_gain': 0.003, 'max_p_nonpos': 0.2, 'n_boot': 2000, 'seed': 0}
print(f'{VERSION}: mode={MODE}, ARM_M={ARM_M}, FOLD0_ARMS={FOLD0_ARMS}, FOLDS_JOBS={FOLDS_JOBS}, '
      f'quota at launch {QUOTA_AT_LAUNCH_H} h, hard stop {HARD_STOP_S / 3600:.2f} h', flush=True)

In [ ]:
# Locate the inputs without walking the competition image tree.
def find_unique_file(name, root=Path('/kaggle/input')):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions', 'cache')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    if len(hits) != 1:
        raise FileNotFoundError(f'expected one {name} under {root}, found {hits}')
    return hits[0]

PATHS = {
    'targets': str(find_unique_file('v06_targets.csv')),
    'folds': str(find_unique_file('v06_folds.csv')),
    'cache_studies': str(find_unique_file('v07_cache_studies.csv')),
}
PATHS['cache_dir'] = str(Path(PATHS['cache_studies']).parent / 'cache')
assert Path(PATHS['cache_dir']).is_dir(), PATHS['cache_dir']
OOF_PATHS = {k: find_unique_file(name) for k, name in OOF_FILES.items()}
V11_GOLD_FOLD0_PATH = find_unique_file(V11_GOLD_FOLD0)
uses_m = MODE == 'smoke' or 'M' in FOLD0_ARMS or any(a == 'M' for a, _ in FOLDS_JOBS)
ARMS['M']['weights'] = str(find_unique_file(MRI_CORE_FILE)) if uses_m else None
print(json.dumps({**PATHS, 'oof': {k: str(v) for k, v in OOF_PATHS.items()}, 'v11_gold_fold0': str(V11_GOLD_FOLD0_PATH),
                  'mri_core': ARMS['M']['weights']}, indent=2))

## Training targets (design 4.3)

For every non-gold study: `(1 - OOF_WEIGHT) * soft + OOF_WEIGHT * v11 OOF`. Checks (as v11): each OOF
file holds exactly the studies of its v06 fold, no gold study has an OOF row, every training study
has one, all values are probabilities. Written to `v13_train_targets.csv`; input hashes go to
`v13_targets_summary.json`.

In [ ]:
import hashlib
import numpy as np
import pandas as pd

targets = pd.read_csv(PATHS['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
folds = pd.read_csv(PATHS['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
cached = pd.read_csv(PATHS['cache_studies'], dtype={'StudyInstanceUID': str})
ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
fold_of = folds.fold.reindex(targets.index)
gold_ids = set(targets.index[targets.is_gold.astype(bool)])

oof_parts, oof_sha = [], {}
for k, path in sorted(OOF_PATHS.items()):
    df = pd.read_csv(path, dtype={'StudyInstanceUID': str})
    assert df.columns.tolist() == ['StudyInstanceUID'] + LABELS, f'{path.name}: unexpected columns'
    assert df.StudyInstanceUID.is_unique, f'{path.name}: duplicate studies'
    expected = {u for u in targets.index if fold_of[u] == k and u in ok}
    assert set(df.StudyInstanceUID) == expected, f'{path.name}: studies differ from v06 fold {k}'
    oof_sha[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    oof_parts.append(df.set_index('StudyInstanceUID'))
oof = pd.concat(oof_parts)
assert oof.index.is_unique and not (set(oof.index) & gold_ids), 'OOF rows must be unique and non-gold'
values = oof[LABELS].to_numpy(np.float64)
assert np.isfinite(values).all() and values.min() >= 0.0 and values.max() <= 1.0, 'OOF values must be probabilities'

pool = [u for u in targets.index if fold_of[u] >= 0 and u in ok]
assert set(pool) == set(oof.index), 'every non-gold training study needs exactly one OOF row'
soft = targets.loc[pool, LABELS].astype(np.float64)
assert np.isfinite(soft.to_numpy()).all(), 'missing soft target'
teacher = oof.loc[pool, LABELS].astype(np.float64)
mixed = (1.0 - OOF_WEIGHT) * soft + OOF_WEIGHT * teacher
mixed.index.name = 'StudyInstanceUID'
PATHS['train_targets'] = str(WORK / 'v13_train_targets.csv')
mixed.to_csv(PATHS['train_targets'])

stats = pd.DataFrame({'soft_mean': soft.mean(), 'oof_mean': teacher.mean(), 'target_mean': mixed.mean(),
                      'soft_std': soft.std(), 'oof_std': teacher.std(), 'target_std': mixed.std()}).round(4)
print(f'training targets: {len(mixed)} studies, gold excluded: {len(gold_ids)}, OOF weight {OOF_WEIGHT}')
print(stats.to_string())
targets_summary = {'version': VERSION, 'teacher': 'v11 arm B OOF (K16)', 'oof_weight': OOF_WEIGHT,
                   'n_train_targets': int(len(mixed)), 'n_gold_excluded': len(gold_ids), 'oof_sha256': oof_sha,
                   'train_targets_sha256': hashlib.sha256(Path(PATHS['train_targets']).read_bytes()).hexdigest(),
                   'per_label': stats.to_dict(orient='index')}
(WORK / 'v13_targets_summary.json').write_text(json.dumps(targets_summary, indent=2))

## Trainer script

Written to disk so that each arm runs in its own process on its own GPU. The notebook is the
source of truth; the script is regenerated on every run.

In [ ]:
%%writefile /kaggle/working/v13_train.py
"""v13 trainer: one arm on one GPU. Configuration comes as a JSON argument.

Derived from the v11 trainer (unchanged: data pipeline, attention head, optimiser, schedule,
augmentation, K16 evaluation, memory probe). Changed for design v13 revision 3.2: the backbone
is built by `build_backbone` (timm models, or the MRI-CORE DINOv2 teacher as a plain ViT-B/16
loaded strictly, with only its last blocks trainable), the input normalisation is configurable
('imagenet' or per-slice 'minmax'), BN layers can be kept in eval mode, and a 'smoke' mode runs
the step-S checks (load, gradients, loss, save/reload) and projects fold and test times.
"""
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
N_SLOTS = 5
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


# ----------------------------------------------------------------------------- data
def load_study(cache_dir, uid):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        return z['jpeg'], z['offsets'], z['slot']


def decode(jpeg, offsets, k):
    return cv2.imdecode(np.frombuffer(jpeg[offsets[k]:offsets[k + 1]].tobytes(), np.uint8),
                        cv2.IMREAD_GRAYSCALE)


def slot_centres(n, k, train, rng):
    if train:
        edges = np.linspace(0, n, k + 1)
        c = np.floor(edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])).astype(int)
    else:
        c = np.linspace(0, n - 1, k).round().astype(int)
    return np.clip(c, 0, n - 1)


def augment(tri, rng):
    # tri: (3, H, W) uint8 triplet; the same geometric transform for all three channels.
    h, w = tri.shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img = img.astype(np.float32) / 255.0
    gamma = rng.uniform(0.85, 1.15)
    img = np.clip(img, 0, 1) ** gamma
    img = img * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    return np.clip(img, 0, 1).transpose(2, 0, 1)


def normalise(x, mode):
    # x: (3, H, W) float in [0, 1]. 'imagenet': ImageNet mean/std (v11). 'minmax': each channel
    # scaled to [0, 1] per slice, as MRI-CORE documents for its encoder input.
    if mode == 'imagenet':
        return (x - MEAN) / STD
    flat = x.reshape(3, -1)
    lo, hi = flat.min(1)[:, None, None], flat.max(1)[:, None, None]
    return np.where(hi > lo, (x - lo) / np.maximum(hi - lo, 1e-6), 0.0).astype(np.float32)


class StudyDataset(Dataset):
    def __init__(self, uids, targets, weights, cache_dir, k, train, img, seed, norm='imagenet'):
        self.uids, self.targets, self.weights = list(uids), targets, weights
        self.cache_dir, self.k, self.train, self.img, self.seed = cache_dir, k, train, img, seed
        self.norm = norm
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        jpeg, offsets, slot = load_study(self.cache_dir, self.uids[i])
        tokens, slots, cache = [], [], {}
        for s in range(N_SLOTS):
            idx = np.flatnonzero(slot == s)
            if len(idx) == 0:
                continue
            n = len(idx)
            for c in slot_centres(n, self.k, self.train, rng):
                chans = []
                for d in (-1, 0, 1):
                    j = int(idx[min(max(c + d, 0), n - 1)])
                    if j not in cache:
                        cache[j] = decode(jpeg, offsets, j)
                    chans.append(cache[j])
                tri = np.stack(chans)
                if tri.shape[1] != self.img:
                    tri = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in tri])
                x = augment(tri, rng) if self.train else tri.astype(np.float32) / 255.0
                tokens.append(normalise(x, self.norm))
                slots.append(s)
        return {'x': np.stack(tokens).astype(np.float32), 'slot': np.asarray(slots, np.int64),
                'y': self.targets[i], 'w': self.weights[i], 'uid': self.uids[i]}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
def build_backbone(cfg, load_weights=True):
    # Returns (backbone without classifier, load report). 'timm': a timm model (ImageNet weights
    # from timm, internet on). 'mricore': the MRI-CORE DINOv2 teacher (teacher.backbone.*) mapped
    # onto timm's plain ViT-B/16 and loaded strictly (design v13 section 11).
    if cfg['kind'] == 'mricore':
        bb = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        report = {'kind': 'mricore'}
        if load_weights:
            sd = torch.load(cfg['weights'], map_location='cpu', weights_only=True)['teacher']
            new = {}
            for key, val in sd.items():
                if not key.startswith('backbone.') or key == 'backbone.mask_token':
                    continue
                k2 = key[len('backbone.'):]
                if k2.startswith('blocks.'):
                    parts = k2.split('.')
                    k2 = '.'.join(['blocks', parts[2]] + parts[3:])
                new[k2] = val
            res = bb.load_state_dict(new, strict=True)
            own = bb.state_dict()
            report.update(missing=list(res.missing_keys), unexpected=list(res.unexpected_keys), n_loaded=len(new),
                          tensors_equal=bool(all(torch.equal(own[key], val) for key, val in new.items())))
            assert not report['missing'] and not report['unexpected'] and report['tensors_equal'], report
    else:
        bb = timm.create_model(cfg['backbone'], pretrained=bool(cfg['pretrained'] and load_weights), num_classes=0)
        report = {'kind': 'timm', 'pretrained': bool(cfg['pretrained'] and load_weights)}
    if cfg.get('train_blocks_from') is not None:
        for p in bb.parameters():
            p.requires_grad = False
        for blk in bb.blocks[cfg['train_blocks_from']:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in bb.norm.parameters():
            p.requires_grad = True
    report['backbone_params'] = int(sum(p.numel() for p in bb.parameters()))
    report['trainable_backbone_params'] = int(sum(p.numel() for p in bb.parameters() if p.requires_grad))
    return bb, report


def set_train_mode(model, bn_eval):
    model.train()
    if bn_eval:
        for mod in model.backbone.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                mod.eval()


def group_grad_norms(model):
    # L2 norm of the gradients of the trainable backbone parameters and of the head.
    out = {}
    for name, params in (('backbone', [p for p in model.backbone.parameters() if p.requires_grad]),
                         ('head', [p for n, p in model.named_parameters() if not n.startswith('backbone.')])):
        grads = [p.grad.detach().float().norm() for p in params if p.grad is not None]
        out[name] = float(torch.stack(grads).norm()) if grads else 0.0
    return out


class Net(nn.Module):
    def __init__(self, backbone, n_out=12, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = backbone
        d = self.backbone.num_features
        self.slot_emb = nn.Embedding(N_SLOTS, d)
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.head_w, std=0.01)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, slot, counts):
        # h: (N, D) tokens of several studies concatenated; counts: tokens per study.
        h = self.norm(h + self.slot_emb(slot))
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()  # (N, 12)
        out, start = [], 0
        for n in counts:
            hs, as_ = h[start:start + n].float(), a[start:start + n]
            wts = torch.softmax(as_, dim=0)                                  # (n, 12)
            pooled = torch.einsum('nk,nd->kd', wts, hs)                      # (12, D)
            out.append((self.drop(pooled) * self.head_w).sum(-1) + self.head_b)
            start += n
        return torch.stack(out)                                              # (B, 12)


def batch_tensors(batch, device):
    x = torch.from_numpy(np.concatenate([b['x'] for b in batch])).to(device, non_blocking=True)
    slot = torch.from_numpy(np.concatenate([b['slot'] for b in batch])).to(device)
    counts = [len(b['slot']) for b in batch]
    return x, slot, counts


# ------------------------------------------------------------------------- metrics
def macro_auc(y_true_bin, y_score):
    per = {}
    for j, name in enumerate(LABELS):
        col = y_true_bin[:, j]
        if col.min() != col.max():
            per[name] = float(roc_auc_score(col, y_score[:, j]))
    return float(np.mean(list(per.values()))) if per else float('nan'), per


def bootstrap_ci(y_true_bin, y_score, n=200, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        s = rng.integers(0, len(y_true_bin), len(y_true_bin))
        m, _ = macro_auc(y_true_bin[s], y_score[s])
        if np.isfinite(m):
            vals.append(m)
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [None, None]


def weighted_bce(logits, y, w):
    # The training loss over a whole set: per-cell BCE with logits times the cell weight, over the weight sum.
    w = torch.from_numpy(np.asarray(w, np.float32))
    cell = F.binary_cross_entropy_with_logits(torch.from_numpy(np.asarray(logits, np.float32)),
                                              torch.from_numpy(np.asarray(y, np.float32)), reduction='none')
    return float((cell * w).sum() / w.sum())


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


@torch.no_grad()
def predict(model, ds, device, batch_tokens, amp):
    # Returns study ids in dataset order and float32 logits.
    model.eval()
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=CFG['num_workers'], collate_fn=collate)
    logits, uids = [], []
    for batch in loader:
        x, slot, counts = batch_tensors(batch, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            h = model.encode(x, chunk=batch_tokens)
            out = model.pool(h, slot, counts)
        logits.append(out.float().cpu().numpy())
        uids += [b['uid'] for b in batch]
    return uids, np.concatenate(logits)


# --------------------------------------------------------------------------- main
CFG = json.loads(sys.argv[1])


def main():
    t_start = time.time()
    arm, fold, out = CFG['arm'], CFG['fold'], Path(CFG['out_dir'])
    out.mkdir(parents=True, exist_ok=True)
    stem = f'{CFG["version"]}_fold_{fold}_{arm}'             # checkpoints: v13_fold_<k>_<arm>_best.pt
    torch.manual_seed(CFG['seed'])
    device = torch.device(CFG['device'] if torch.cuda.is_available() or CFG['device'] == 'cpu' else 'cpu')
    amp = device.type == 'cuda'

    targets = pd.read_csv(CFG['targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    fit = pd.read_csv(CFG['train_targets'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    folds = pd.read_csv(CFG['folds'], dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
    cached = pd.read_csv(CFG['cache_studies'], dtype={'StudyInstanceUID': str})
    ok = set(cached[cached.error.fillna('') == ''].StudyInstanceUID)
    fold_of = folds.fold.reindex(targets.index)
    train_ids = [u for u in targets.index if fold_of[u] >= 0 and fold_of[u] != fold and u in ok]
    val_ids = [u for u in targets.index if fold_of[u] == fold and u in ok]
    gold_ids = [u for u in targets.index if bool(targets.loc[u, 'is_gold']) and u in ok]
    assert not set(gold_ids) & (set(train_ids) | set(val_ids)), 'gold studies are never trained or validated on'
    Y_soft = targets[LABELS].to_numpy(np.float32)
    Y_fit = fit.reindex(targets.index)[LABELS].to_numpy(np.float32)    # training target; NaN for gold
    W = targets[[f'{c}__weight' for c in LABELS]].to_numpy(np.float32)
    pos = {u: i for i, u in enumerate(targets.index)}
    pick = lambda ids, m: m[[pos[u] for u in ids]]
    assert np.isfinite(pick(train_ids + val_ids, Y_fit)).all(), 'a training or validation study has no target'
    n_probe = min(int(CFG['train_probe_n']), len(train_ids))
    probe_ids = sorted(np.random.default_rng(CFG['seed']).choice(sorted(train_ids), n_probe, replace=False).tolist()) \
        if n_probe else []
    log(f'arm {arm} fold {fold}: train {len(train_ids)}, val {len(val_ids)}, gold {len(gold_ids)}, '
        f'train probe {len(probe_ids)}, device {device}')

    k = CFG['k_train']
    train_ds = StudyDataset(train_ids, pick(train_ids, Y_fit), pick(train_ids, W), CFG['cache_dir'], k, True, CFG['img'], CFG['seed'], CFG['norm'])
    mb = CFG['micro_batch_studies']
    accum = max(1, CFG['effective_batch_studies'] // mb)
    backbone, load_report = build_backbone(CFG)
    model = Net(backbone).to(device)
    log(f'arm {arm}: backbone {CFG["backbone"]} ({CFG["kind"]}), load {json.dumps(load_report)}')
    if CFG['grad_checkpointing']:
        model.backbone.set_grad_checkpointing(True)
    probe = {'grad_checkpointing': bool(CFG['grad_checkpointing']), 'micro_batch_studies': mb, 'attempts': []}
    if device.type == 'cuda':
        while True:
            n_tok = mb * N_SLOTS * k
            try:
                xx = torch.randn(n_tok, 3, CFG['img'], CFG['img'], device=device)
                ss = torch.arange(n_tok, device=device) % N_SLOTS
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    lo = model.pool(model.encode(xx), ss, [N_SLOTS * k] * mb)
                lo.float().sum().backward()
                model.zero_grad(set_to_none=True)
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': True,
                                          'peak_reserved_gb': round(torch.cuda.max_memory_reserved() / 1e9, 2)})
                del xx, ss, lo
                torch.cuda.empty_cache()
                break
            except torch.OutOfMemoryError:
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                probe['attempts'].append({'tokens': n_tok, 'ckpt': probe['grad_checkpointing'], 'ok': False})
                if not probe['grad_checkpointing']:
                    model.backbone.set_grad_checkpointing(True)
                    probe['grad_checkpointing'] = True
                elif mb > 1:
                    mb = mb // 2
                    probe['micro_batch_studies'] = mb
                else:
                    raise
        accum = max(1, CFG['effective_batch_studies'] // mb)
        log(f'arm {arm}: memory probe {probe}')
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    opt = torch.optim.AdamW([{'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': CFG['lr_backbone']},
                             {'params': head_params, 'lr': CFG['lr_head']}], weight_decay=CFG['weight_decay'])
    base_lrs = [CFG['lr_backbone'], CFG['lr_head']]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    steps_per_epoch = math.ceil(len(train_ids) / (mb * accum))
    total_steps = steps_per_epoch * CFG['epochs']
    warm = steps_per_epoch * CFG['warmup_epochs']

    def set_lr(step):
        if CFG['mode'] == 'smoke':   # step S: 20 warm-up steps, then constant (design v13 4.2)
            f = min(1.0, (step + 1) / 20)
        else:
            f = step / max(1, warm) if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
        for g, lr in zip(opt.param_groups, base_lrs):
            g['lr'] = lr * f

    receipt = {'version': CFG['version'], 'arm': arm, 'fold': fold, 'status': 'running',
               'epochs': CFG['epochs'], 'lr_backbone': CFG['lr_backbone'], 'lr_head': CFG['lr_head'],
               'k_train': k, 'k_infer': CFG['k_infer'], 'time_limit_s': CFG['time_limit_s'],
               'memory_probe': probe, 'n_train': len(train_ids), 'n_val': len(val_ids), 'n_gold': len(gold_ids),
               'n_train_probe': len(probe_ids),
               'train_probe_sha1': hashlib.sha1('\n'.join(probe_ids).encode()).hexdigest()[:12],
               'train_targets_sha256': hashlib.sha256(Path(CFG['train_targets']).read_bytes()).hexdigest(),
               'micro_batch_studies': mb, 'accum': accum, 'tokens_per_study_train': N_SLOTS * k,
               'backbone': CFG['backbone'], 'kind': CFG['kind'], 'img': CFG['img'], 'norm': CFG['norm'],
               'bn_eval': bool(CFG.get('bn_eval', False)), 'train_blocks_from': CFG.get('train_blocks_from'),
               'load_report': load_report, 'device_name': torch.cuda.get_device_name(0) if amp else 'cpu',
               'torch': torch.__version__, 'timm': timm.__version__,
               'epoch_eval': {}}
    write = lambda: (out / f'{CFG["version"]}_receipt_fold{fold}_{arm}.json').write_text(json.dumps(receipt, indent=2))

    def time_up():
        return time.time() - t_start > CFG['time_limit_s']

    def save_state(tag, epoch, step):
        torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'step': step, 'cfg': CFG}, out / f'{stem}_{tag}.pt')

    def run_eval(ids, y):
        # Same checkpoint, no augmentation, K_infer evenly spaced centres per slot.
        ds = StudyDataset(ids, pick(ids, y), pick(ids, W), CFG['cache_dir'], CFG['k_infer'], False, CFG['img'], CFG['seed'], CFG['norm'])
        t = time.time()
        uids, z = predict(model, ds, device, CFG['eval_batch_tokens'], amp)
        assert uids == list(ids)
        return z, time.time() - t

    def eval_record(ids, z, seconds):
        # AUC against the soft target rounded at 0.5; loss against the training target and the soft target.
        m, per = macro_auc((pick(ids, Y_soft) >= 0.5).astype(np.float32), sigmoid(z))
        return {'auc': m, 'loss': weighted_bce(z, pick(ids, Y_fit), pick(ids, W)),
                'loss_soft': weighted_bce(z, pick(ids, Y_soft), pick(ids, W)), 'seconds': round(seconds, 1)}, per

    step, micro, best, best_epoch, best_per, stopped = 0, 0, -1.0, -1, {}, False
    t_data = t_compute = 0.0
    n_images = 0
    torch.cuda.reset_peak_memory_stats() if amp else None
    epoch_seconds = []
    loss_trace, grad_check, grad_attempts = [], None, 0
    n_loop = (max(1, math.ceil(CFG['smoke_steps'] / max(1, len(train_ids) // mb))) if CFG['mode'] == 'smoke'
              else CFG['epochs'])   # step S runs until smoke_steps micro-steps, across epochs if needed
    for epoch in range(n_loop):
        train_ds.epoch = epoch
        loader = DataLoader(train_ds, batch_size=mb, shuffle=True, num_workers=CFG['num_workers'],
                            collate_fn=collate, drop_last=True, persistent_workers=False,
                            generator=torch.Generator().manual_seed(CFG['seed'] + epoch))
        set_train_mode(model, CFG.get('bn_eval', False))
        opt.zero_grad(set_to_none=True)
        loss_sum, loss_n = 0.0, 0
        t0 = time.time()
        for batch in loader:
            t1 = time.time()
            t_data += t1 - t0
            x, slot, counts = batch_tensors(batch, device)
            y = torch.from_numpy(np.stack([b['y'] for b in batch])).to(device)
            w = torch.from_numpy(np.stack([b['w'] for b in batch])).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model.pool(model.encode(x), slot, counts)
            loss = (F.binary_cross_entropy_with_logits(logits.float(), y, reduction='none') * w).sum() / w.sum()
            scaler.scale(loss / accum).backward()
            micro += 1
            n_images += x.shape[0]
            loss_sum += loss.item()
            loss_n += 1
            loss_trace.append(loss.item())
            if micro % accum == 0:
                set_lr(step)
                scaler.unscale_(opt)
                if grad_check is None and CFG['mode'] == 'smoke':
                    grad_attempts += 1
                    norms = group_grad_norms(model)
                    if all(np.isfinite(v) for v in norms.values()):   # an overflowed step is skipped by the scaler
                        grad_check = norms
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                step += 1
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            t_compute += t0 - t1
            if micro % 50 == 0:
                log(f'arm {arm} epoch {epoch} micro {micro} loss {loss.item():.4f} '
                    f'img/s {n_images / max(t_data + t_compute, 1e-9):.1f} data share {t_data / max(t_data + t_compute, 1e-9):.2f}')
            if time_up():
                stopped = True
                break
            if CFG['mode'] == 'smoke' and micro >= CFG['smoke_steps']:
                break
        epoch_seconds.append(round(time.time() - t_start, 1))
        if CFG['mode'] == 'smoke' and micro >= CFG['smoke_steps']:
            write()
            break
        receipt['elapsed_at_epoch_end'] = epoch_seconds
        if stopped:
            save_state('last', epoch, step)
            receipt.update(status='stopped_early', stopped_epoch=epoch, stopped_step=step)
            break
        if CFG['mode'] != 'smoke' and epoch + 1 >= CFG['eval_from_epoch']:
            z_val, s_val = run_eval(val_ids, Y_fit)
            rec_val, per_val = eval_record(val_ids, z_val, s_val)
            rec = {'val': rec_val, 'train_batch_loss_mean': loss_sum / max(loss_n, 1), 'train_probe': None}
            if probe_ids:
                z_tr, s_tr = run_eval(probe_ids, Y_fit)
                rec['train_probe'], _ = eval_record(probe_ids, z_tr, s_tr)
            receipt['epoch_eval'][str(epoch)] = rec
            tp = rec['train_probe'] or {}
            log(f'arm {arm} epoch {epoch} K{CFG["k_infer"]}: val AUC {rec_val["auc"]:.4f} loss {rec_val["loss"]:.4f} '
                f'({rec_val["seconds"]:.0f} s) | train probe AUC {tp.get("auc", float("nan")):.4f} '
                f'loss {tp.get("loss", float("nan")):.4f} ({tp.get("seconds", 0):.0f} s) | batch loss {rec["train_batch_loss_mean"]:.4f}')
            if rec_val['auc'] > best:
                best, best_epoch, best_per = rec_val['auc'], epoch, per_val
                torch.save({'model': model.state_dict(), 'epoch': epoch, 'cfg': CFG}, out / f'{stem}_best.pt')
                pd.DataFrame(sigmoid(z_val), columns=LABELS, index=pd.Index(val_ids, name='StudyInstanceUID')).to_csv(
                    out / f'{CFG["version"]}_oof_fold{fold}_{arm}_k{CFG["k_infer"]}.csv')
        write()

    elapsed = t_data + t_compute
    receipt.update(train_images=n_images, train_seconds=round(elapsed, 1),
                   train_images_per_s=round(n_images / max(elapsed, 1e-9), 1),
                   data_share=round(t_data / max(elapsed, 1e-9), 3), micro_steps=micro, optimizer_steps=step)
    if amp:
        receipt.update(peak_allocated_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                       peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 2))

    if CFG['mode'] == 'smoke':
        # Step S (design v13 4.2): one K16 validation pass, the four checks, and projections.
        z_val, s_val = run_eval(val_ids, Y_fit)
        m_val, _ = macro_auc((pick(val_ids, Y_soft) >= 0.5).astype(np.float32), sigmoid(z_val))
        ckpt = out / f'{stem}_smoke.pt'
        torch.save({'model': model.state_dict(), 'cfg': CFG}, ckpt)
        bb2, _ = build_backbone(CFG, load_weights=False)
        model2 = Net(bb2).to(device)
        model2.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False)['model'])
        two = val_ids[:2]
        ds2 = StudyDataset(two, pick(two, Y_soft), pick(two, W), CFG['cache_dir'], CFG['k_infer'], False,
                           CFG['img'], CFG['seed'], CFG['norm'])
        _, za = predict(model, ds2, device, CFG['eval_batch_tokens'], False)
        _, zb = predict(model2, ds2, device, CFG['eval_batch_tokens'], False)
        reload_diff = float(np.abs(za - zb).max())
        ckpt.unlink()
        win = max(1, min(50, len(loss_trace) // 2))
        loss_first, loss_last = float(np.mean(loss_trace[:win])), float(np.mean(loss_trace[-win:]))
        sec_per_micro = elapsed / max(micro, 1)
        n_evals = CFG['epochs'] - CFG['eval_from_epoch'] + 1
        fold_s = (sec_per_micro * (len(train_ids) // mb) * CFG['epochs'] + n_evals * s_val
                  + s_val * len(gold_ids) / max(len(val_ids), 1) + 180)
        test_leg_s = s_val / max(len(val_ids), 1) * CFG['n_test_studies'] * 5 / 2   # five folds, two GPUs
        lr = receipt['load_report']
        checks = {
            'load': bool(lr.get('pretrained', True) if lr['kind'] == 'timm'
                         else (not lr['missing'] and not lr['unexpected'] and lr['tensors_equal'])),
            'gradients': bool(grad_check is not None and all(np.isfinite(v) and v > 0 for v in grad_check.values())),
            'loss_decreases': bool(loss_last < loss_first),
            'reload': bool(reload_diff < 1e-5),
        }
        receipt.update(status='smoke_done', smoke_micro_steps=micro, smoke_val_macro_auc=m_val,
                       val_seconds_k16=round(s_val, 1), seconds_per_micro_step=round(sec_per_micro, 3),
                       projected_fold_hours=round(fold_s / 3600, 2), projected_test_leg_hours=round(test_leg_s / 3600, 2),
                       loss_first_mean=loss_first, loss_last_mean=loss_last, loss_window=win,
                       grad_norms=grad_check, grad_attempts=grad_attempts, reload_max_diff=reload_diff,
                       checks=checks, checks_passed=bool(all(checks.values())),
                       total_seconds=round(time.time() - t_start, 1))
        write()
        log(json.dumps({k_: v for k_, v in receipt.items() if k_ != 'epoch_eval'}, default=str))
        return

    if best_epoch < 0:
        # No finite validation AUC (or stopped before the first evaluation): keep the last weights
        # and say so instead of leaving the receipt in 'running'.
        save_state('last', epoch, step)
        if receipt['status'] == 'running':
            receipt['status'] = 'no_valid_eval'
        receipt['total_seconds'] = round(time.time() - t_start, 1)
        write()
        log(json.dumps({k_: v for k_, v in receipt.items() if k_ != 'epoch_eval'}))
        return
    # Gold evaluation of the best checkpoint (also after an early stop, for the record only).
    model.load_state_dict(torch.load(out / f'{stem}_best.pt', map_location=device)['model'])
    z_gold, _ = run_eval(gold_ids, Y_soft)
    p_gold = sigmoid(z_gold)
    gyb = targets.loc[gold_ids, [f'{c}__gold' for c in LABELS]].to_numpy(np.float32)
    gm, gper = macro_auc(gyb, p_gold)
    pd.DataFrame(p_gold, columns=LABELS, index=pd.Index(gold_ids, name='StudyInstanceUID')).to_csv(
        out / f'{CFG["version"]}_gold_fold{fold}_{arm}_k{CFG["k_infer"]}.csv')
    receipt.update(status='done' if receipt['status'] == 'running' else 'stopped_early_evaluated',
                   best_epoch=best_epoch, val_macro_k16=best, val_per_label_k16=best_per,
                   gold_macro_k16=gm, gold_ci=bootstrap_ci(gyb, p_gold), gold_per_label=gper,
                   total_seconds=round(time.time() - t_start, 1))
    write()
    log(json.dumps({k_: v for k_, v in receipt.items() if k_ not in ('epoch_eval', 'val_per_label_k16', 'gold_per_label')}))


if __name__ == '__main__':
    main()

## Launch helpers, step S and step F0

One process per arm, each on its own GPU. Time limits follow the budget formula (design 4.1). A
process is stopped and counts as incomplete if it is still running 5 minutes before the hard stop
or `HANG_GRACE_S` after its own limit. A stale receipt is removed at launch; a non-zero exit code
marks the arm `failed`.

In [ ]:
HANG_GRACE_S = 3600

def remaining_s():
    return HARD_STOP_S - (time.time() - NOTEBOOK_T0)

def receipt_path(tag, fold, arm):
    return WORK / tag / f'v13_receipt_fold{fold}_{arm}.json'

def arm_cfg(arm):
    return {k: v for k, v in ARMS[arm].items() if k != 'projected_fold_h'}

def launch(arm, fold, gpu, time_limit_s, mode):
    tag = f'fold{fold}_{arm}'
    receipt_path(tag, fold, arm).unlink(missing_ok=True)
    cfg = {**COMMON, **PATHS, **arm_cfg(arm), 'mode': mode, 'arm': arm, 'fold': fold,
           'time_limit_s': time_limit_s, 'out_dir': str(WORK / tag)}
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    logf = open(WORK / f'v13_log_{tag}.txt', 'w')
    proc = subprocess.Popen([sys.executable, '/kaggle/working/v13_train.py', json.dumps(cfg)],
                            env=env, stdout=logf, stderr=subprocess.STDOUT)
    print(f'launched {tag} ({mode}) on GPU {gpu} at {(time.time() - NOTEBOOK_T0) / 3600:.2f} h, '
          f'time limit {time_limit_s / 3600:.2f} h', flush=True)
    return {'tag': tag, 'arm': arm, 'fold': fold, 'proc': proc, 'logf': logf, 'killed': None,
            't0': time.time(), 'limit_s': time_limit_s}

def stop_if_overdue(job):
    if job['proc'].poll() is not None:
        return
    if remaining_s() < 5 * 60:
        job['killed'] = 'killed_at_deadline'
    elif time.time() - job['t0'] > job['limit_s'] + HANG_GRACE_S:
        job['killed'] = 'killed_hang_guard'
    else:
        return
    job['proc'].terminate()
    job['proc'].wait()
    print(f'{job["tag"]} stopped: {job["killed"]}', flush=True)

def collect(job):
    job['logf'].close()
    f = receipt_path(job['tag'], job['fold'], job['arm'])
    r = json.loads(f.read_text()) if f.is_file() else {'status': 'missing'}
    r['returncode'] = job['proc'].returncode
    if job['killed']:
        r['status_reported'], r['status'] = r.get('status'), job['killed']
    elif job['proc'].returncode != 0:
        r['status_reported'], r['status'] = r.get('status'), 'failed'
    print(f'{job["tag"]}: exit code {job["proc"].returncode}, status {r.get("status")}', flush=True)
    print(''.join(open(WORK / f'v13_log_{job["tag"]}.txt').readlines()[-6:]), flush=True)
    return r

def wait_all(jobs):
    out, pending = {}, list(jobs)
    while pending:
        for job in list(pending):
            stop_if_overdue(job)
            if job['proc'].poll() is not None:
                out[job['arm']] = collect(job)
                pending.remove(job)
        if pending:
            time.sleep(30)
    return out

def admissible(projected_h):
    # Budget formula (design 4.1): projected x 1.15 + 0.25 h must fit in the time left.
    return projected_h is not None and projected_h * GATE_S['margin'] + GATE_S['overhead_h'] <= remaining_s() / 3600

stage = {}
if MODE == 'smoke':
    jobs = [launch(a, 0, g, min(1.0 * 3600, remaining_s() - 15 * 60), 'smoke') for g, a in enumerate(('M', 'E'))]
    stage = wait_all(jobs)
elif MODE == 'fold0_arms':
    jobs = []
    for a in FOLD0_ARMS:
        assert a == 'N' or a == ARM_M, f'arm {a} is not N or the admitted ARM_M {ARM_M}'
        proj = ARMS[a]['projected_fold_h']
        if not admissible(proj):
            print(f'arm {a} not admitted: projected {proj} h, {remaining_s() / 3600:.2f} h left', flush=True)
            continue
        jobs.append(launch(a, 0, len(jobs), min(proj * GATE_S['margin'] * 3600, remaining_s() - 15 * 60), 'fold'))
    stage = wait_all(jobs)
print(f'notebook elapsed {(time.time() - NOTEBOOK_T0) / 3600:.2f} h', flush=True)

## Step S decision (`smoke`)

Admission rule (design 4.2): all four checks pass, peak reserved memory < 14 GB, and the projected
fold time satisfies the budget formula for the quota expected at the F0 launch (`QUOTA_AT_F0_H`),
i.e. `projected x 1.15 + 0.25 <= QUOTA_AT_F0_H - 0.4`. MRI-CORE (M) is preferred, else
EfficientNet-B3 (E), else F0 runs N alone. The projected test-leg time feeds the inference gate
(design 4.6). The final check is repeated with the real quota before the F0 push.

In [ ]:
if MODE == 'smoke':
    limit_h = (QUOTA_AT_F0_H - GATE_S['reserve_h'] - GATE_S['overhead_h']) / GATE_S['margin']
    smoke = {'quota_at_f0_h': QUOTA_AT_F0_H, 'admission_limit_h': round(limit_h, 3),
             'n_projected_h': ARMS['N']['projected_fold_h'], 'n_admissible': ARMS['N']['projected_fold_h'] <= limit_h}
    for a in ('M', 'E'):
        r = stage.get(a, {})
        ok_mem = (r.get('peak_reserved_gb') or 0) < GATE_S['max_reserved_gb']
        ok = (r.get('status') == 'smoke_done' and bool(r.get('checks_passed')) and ok_mem
              and (r.get('projected_fold_hours') or 99) <= limit_h)
        smoke[a] = {'admitted': bool(ok), 'status': r.get('status'), 'checks': r.get('checks'),
                    'peak_reserved_gb': r.get('peak_reserved_gb'), 'projected_fold_hours': r.get('projected_fold_hours'),
                    'projected_test_leg_hours': r.get('projected_test_leg_hours'),
                    'train_images_per_s': r.get('train_images_per_s'), 'data_share': r.get('data_share'),
                    'val_seconds_k16': r.get('val_seconds_k16'), 'smoke_val_macro_auc': r.get('smoke_val_macro_auc'),
                    'load_report': r.get('load_report')}
    smoke['arm_m'] = 'M' if smoke['M']['admitted'] else ('E' if smoke['E']['admitted'] else None)
    print(json.dumps(smoke, indent=2, default=float))
    (WORK / 'v13_decision_smoke.json').write_text(json.dumps(smoke, indent=2, default=float))

## Gate F0 (`fold0_arms`, design 4.4)

Arm states: done, completed-fail (gold < 0.900), incomplete (anything else), not-run. If an arm is
incomplete the decision is `pending_rerun` (one technical rerun in the next window). Otherwise:
base = N if gold(N) - gold(v11 fold 0) >= +0.003 with P(gain <= 0) <= 0.2, else v11 fold 0;
candidate C = base + M if the rank mean of base and M beats base by >= +0.003 with P <= 0.2,
else base. C = v11 fold 0 alone means stop. Paired bootstrap: 2,000 study resamples, seed 0,
single-class labels skipped. The total gain of C over v11 fold 0 is reported with its CI. These
are exploratory screens (design 4.1).

In [ ]:
from sklearn.metrics import roc_auc_score

def macro_auc_gold(y, p):
    vals = [roc_auc_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(vals)) if vals else float('nan')

def paired_gain(y, a, b, n_boot, seed):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        s = rng.integers(0, len(y), len(y))
        d.append(macro_auc_gold(y[s], b[s]) - macro_auc_gold(y[s], a[s]))
    d = np.asarray(d)
    return {'gain': macro_auc_gold(y, b) - macro_auc_gold(y, a), 'ci': [float(np.percentile(d, 2.5)),
            float(np.percentile(d, 97.5))], 'p_nonpos': float((d <= 0).mean())}

def rank_mean(*preds):
    ranks = [pd.DataFrame(p).rank(pct=True).to_numpy() for p in preds]
    return pd.DataFrame(sum(ranks) / len(ranks)).rank(pct=True).to_numpy()

def gate_f0(receipts, gold_preds, v11_gold, y, arm_m, cfg=GATE_F0):
    states = {}
    for a in ('N', arm_m):
        if a is None:
            continue
        r = receipts.get(a)
        if r is None:
            states[a] = 'not-run'
        elif r.get('status') != 'done':
            states[a] = 'incomplete'
        else:
            states[a] = 'done' if (r.get('gold_macro_k16') or 0) >= cfg['sanity_gold'] else 'completed-fail'
    d = {'states': states, 'arm_m': arm_m, 'v11_fold0_gold': macro_auc_gold(y, v11_gold)}
    pending = [a for a, s in states.items() if s == 'incomplete']
    if pending:
        d.update(decision='pending_rerun', rerun=pending)
        return d
    passes = lambda g: g['gain'] >= cfg['min_gain'] and g['p_nonpos'] <= cfg['max_p_nonpos']
    base, base_pred = 'v11', v11_gold
    if states.get('N') == 'done':
        d['n_vs_v11'] = paired_gain(y, v11_gold, gold_preds['N'], cfg['n_boot'], cfg['seed'])
        if passes(d['n_vs_v11']):
            base, base_pred = 'N', gold_preds['N']
    cand, cand_pred = base, base_pred
    if arm_m is not None and states.get(arm_m) == 'done':
        comb = rank_mean(base_pred, gold_preds[arm_m])
        d['m_adds_to_base'] = paired_gain(y, base_pred, comb, cfg['n_boot'], cfg['seed'])
        if passes(d['m_adds_to_base']):
            cand, cand_pred = f'{base}+{arm_m}', comb
    d['base'], d['candidate'] = base, cand
    if cand == 'v11':
        d['decision'] = 'stop'
    else:
        d['decision'] = 'step_D'
        d['total_vs_v11'] = paired_gain(y, v11_gold, cand_pred, cfg['n_boot'], cfg['seed'])
    return d

def load_gold_pred(path, ids):
    return pd.read_csv(path, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[ids, LABELS].to_numpy(float)

if MODE == 'fold0_arms':
    gold_list = sorted(gold_ids)
    y_gold = targets.loc[gold_list, [f'{c}__gold' for c in LABELS]].to_numpy(float)
    preds = {}
    for a, r in stage.items():
        f = WORK / f'fold0_{a}' / f'v13_gold_fold0_{a}_k16.csv'
        if r.get('status') == 'done' and f.is_file():
            preds[a] = load_gold_pred(f, gold_list)
    decision = gate_f0(stage, preds, load_gold_pred(V11_GOLD_FOLD0_PATH, gold_list), y_gold, ARM_M)
    decision['receipts'] = {a: {k: r.get(k) for k in ('status', 'best_epoch', 'val_macro_k16', 'gold_macro_k16',
                                                        'gold_ci', 'total_seconds')} for a, r in stage.items()}
    print(json.dumps(decision, indent=2, default=float))
    (WORK / 'v13_decision_f0.json').write_text(json.dumps(decision, indent=2, default=float))

## Fold queue (`folds`, design 4.6)

Runs the (arm, fold) jobs in `FOLDS_JOBS` over the two GPUs. A job starts only if its arm's
projected fold time satisfies the budget formula with the time left; otherwise it is skipped and
listed (it then moves to the next window). For N+M, list the jobs so that each session pairs one
N fold with one M fold.

In [ ]:
queue_summary = {}
if MODE == 'folds':
    queue, running, skipped = list(FOLDS_JOBS), {}, []
    while queue or running:
        for gpu in (0, 1):
            if gpu in running or not queue:
                continue
            arm, fold = queue[0]
            proj = ARMS[arm]['projected_fold_h']
            if not admissible(proj):
                skipped += queue
                print(f'not enough time for {queue}: {remaining_s() / 3600:.2f} h left', flush=True)
                queue = []
                break
            queue.pop(0)
            running[gpu] = launch(arm, fold, gpu, min(proj * GATE_S['margin'] * 3600, remaining_s() - 15 * 60), 'fold')
        for gpu, job in list(running.items()):
            stop_if_overdue(job)
            if job['proc'].poll() is None:
                continue
            r = collect(job)
            queue_summary[job['tag']] = {k: r.get(k) for k in ('status', 'best_epoch', 'val_macro_k16', 'gold_macro_k16',
                                                                'gold_ci', 'total_seconds', 'val_per_label_k16')}
            del running[gpu]
        if running:
            time.sleep(30)
    queue_summary['_jobs'] = FOLDS_JOBS
    queue_summary['_skipped'] = skipped
    queue_summary['_notebook_hours'] = round((time.time() - NOTEBOOK_T0) / 3600, 2)
    (WORK / 'v13_folds_summary.json').write_text(json.dumps(queue_summary, indent=2, default=float))
    print(json.dumps(queue_summary, indent=2, default=float))